In [ ]:
import pandas as pd
import numpy as np

In [ ]:
from datasets import load_dataset

In [ ]:
data = load_dataset("findnitai/english-to-hinglish")

README.md:   0%|          | 0.00/367 [00:00<?, ?B/s]

hinglish_upload_v1.json: reconstructing file:   0%|          |  0.00B / 27.1MB            

hinglish_upload_v1.json: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/189102 [00:00<?, ? examples/s]

In [ ]:
data

DatasetDict({
    train: Dataset({
        features: ['translation'],
        num_rows: 189102
    })
})

In [ ]:
train_eng = [text['en'] for text in data['train'][0:184000]['translation']]
train_hing = [text['hi_ng'] for text in data['train'][0:184000]['translation']]

In [ ]:
train_hing[9]

'Movie 2015 mein banee thee'

In [ ]:
train_eng[9]

'the movie was made in 2015'

In [ ]:
test_eng = [text['en'] for text in data['train'][184001:189102]['translation']]
test_hing = [text['hi_ng'] for text in data['train'][184001:189102]['translation']]

In [ ]:
df_train = pd.DataFrame({'English': train_eng, 'Hinglish': train_hing})

df_test = pd.DataFrame({'English': test_eng, 'Hinglish': test_hing})

In [ ]:
df_train.head()

,English,Hinglish
0,What's the name of the movie,film ka kya naam hai
1,"Hi, the rotten tomatoes score is great but the...","namaste, sada hua tomatoes score mahaan hai, l..."
2,Do you think you will like the movie,kya aapako lagata hai ki aapako film pasand aa...
3,What kind of movie is it,yah kis tarah kee philm hai
4,when was the movie made?,film kab banee thee?


In [ ]:
print(df_train.head())
print(df_train.columns)
print(df_train.shape)
print(df_train.info())

                                             English  \
0                       What's the name of the movie   
1  Hi, the rotten tomatoes score is great but the...   
2               Do you think you will like the movie   
3                           What kind of movie is it   
4                           when was the movie made?   

                                            Hinglish  
0                               film ka kya naam hai  
1  namaste, sada hua tomatoes score mahaan hai, l...  
2  kya aapako lagata hai ki aapako film pasand aa...  
3                        yah kis tarah kee philm hai  
4                              film  kab banee thee?  
Index(['English', 'Hinglish'], dtype='object')
(184000, 2)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 184000 entries, 0 to 183999
Data columns (total 2 columns):
 #   Column    Non-Null Count   Dtype 
---  ------    --------------   ----- 
 0   English   184000 non-null  object
 1   Hinglish  184000 non-null  object
dtypes: 

In [ ]:
import pandas as pd

# Original DataFrame ki copy
data = df_train[["English", "Hinglish"]].copy()

# Missing values remove karo
data = data.dropna(subset=["English", "Hinglish"])

# Extra spaces remove karo
data["English"] = data["English"].str.strip()
data["Hinglish"] = data["Hinglish"].str.strip()

# Empty sentences remove karo
data = data[
    (data["English"] != "") &
    (data["Hinglish"] != "")
]

# Exact duplicate pairs remove karo
data = data.drop_duplicates(
    subset=["English", "Hinglish"]
).reset_index(drop=True)

print("Clean dataset:", data.shape)
display(data.head())

Clean dataset: (161234, 2)


,English,Hinglish
0,What's the name of the movie,film ka kya naam hai
1,"Hi, the rotten tomatoes score is great but the...","namaste, sada hua tomatoes score mahaan hai, l..."
2,Do you think you will like the movie,kya aapako lagata hai ki aapako film pasand aa...
3,What kind of movie is it,yah kis tarah kee philm hai
4,when was the movie made?,film kab banee thee?


In [ ]:
from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(
    data,
    test_size=0.20,
    random_state=42
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=42
)

print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))

Train: 128987
Validation: 16123
Test: 16124


In [ ]:
!pip install -q transformers datasets sentencepiece accelerate sacrebleu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.7/129.7 kB 4.6 MB/s eta 0:00:00


In [ ]:
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM
)

MODEL_NAME = "google-t5/t5-small"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME
)

print("Tokenizer and model loaded!")

config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.32k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  242MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Tokenizer and model loaded!


In [ ]:
from datasets import Dataset

train_ds = Dataset.from_pandas(
    train_df.reset_index(drop=True)
)

val_ds = Dataset.from_pandas(
    val_df.reset_index(drop=True)
)

test_ds = Dataset.from_pandas(
    test_df.reset_index(drop=True)
)

MAX_LENGTH = 128

def preprocess(batch):
    inputs = [
        "translate English to Hinglish: " + text
        for text in batch["English"]
    ]

    model_inputs = tokenizer(
        inputs,
        max_length=MAX_LENGTH,
        truncation=True
    )

    targets = tokenizer(
        text_target=batch["Hinglish"],
        max_length=MAX_LENGTH,
        truncation=True
    )

    model_inputs["labels"] = targets["input_ids"]

    return model_inputs


tokenized_train = train_ds.map(
    preprocess,
    batched=True,
    remove_columns=train_ds.column_names
)

tokenized_val = val_ds.map(
    preprocess,
    batched=True,
    remove_columns=val_ds.column_names
)

print(tokenized_train[0].keys())

Map:   0%|          | 0/128987 [00:00<?, ? examples/s]

Map:   0%|          | 0/16123 [00:00<?, ? examples/s]

dict_keys(['input_ids', 'attention_mask', 'labels'])


In [ ]:
import torch

print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print("Training samples:", len(tokenized_train))
print("Validation samples:", len(tokenized_val))

print(tokenized_train[0].keys())

GPU available: True
GPU: Tesla T4
Training samples: 128987
Validation samples: 16123
dict_keys(['input_ids', 'attention_mask', 'labels'])


In [ ]:
from transformers import (
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments
)

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model
)

training_args = Seq2SeqTrainingArguments(
    output_dir="./hinglish_checkpoints",

    learning_rate=3e-4,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,

    gradient_accumulation_steps=2,
    num_train_epochs=3,

    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,

    logging_steps=100,
    fp16=torch.cuda.is_available(),
    report_to="none"
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    processing_class=tokenizer,
    data_collator=data_collator
)

trainer.train()

Epoch,Training Loss,Validation Loss
1,1.158771,0.506776
2,0.963250,0.444972
3,0.818471,0.426526


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=24186, training_loss=1.2098015813844076, metrics={'train_runtime': 3653.3343, 'train_samples_per_second': 105.92, 'train_steps_per_second': 6.62, 'total_flos': 3076383117606912.0, 'train_loss': 1.2098015813844076, 'epoch': 3.0})

In [ ]:
SAVE_PATH = "./hinglish_translator_model"

trainer.save_model(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)

print("Model saved successfully!")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved successfully!


In [ ]:

import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = trainer.model
tokenizer = trainer.processing_class

model.to(device)
model.eval()

def translate_to_hinglish(text):
    inputs = tokenizer(
        "translate English to Hinglish: " + text,
        return_tensors="pt",
        max_length=128,
        truncation=True
    ).to(device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=128,
            num_beams=4,
            early_stopping=True
        )

    return tokenizer.decode(
        output_ids[0],
        skip_special_tokens=True
    )

test_sentences = [
    "What is your name?",
    "How are you doing today?",
    "I want to learn machine learning.",
    "Can you help me with this problem?",
    "Do you think this movie is good?"
]

for sentence in test_sentences:
    print("English:", sentence)
    print("Hinglish:", translate_to_hinglish(sentence))
    print("-" * 50)


English: What is your name?
Hinglish: tumhara naam kya hai?
--------------------------------------------------
English: How are you doing today?
Hinglish: aaj kaise kar raha hai?
--------------------------------------------------
English: I want to learn machine learning.
Hinglish: mai machine learning karna chahta hoon
--------------------------------------------------
English: Can you help me with this problem?
Hinglish: Kya aap muje is problem ke sath help karsakte hai?
--------------------------------------------------
English: Do you think this movie is good?
Hinglish: kya tumhe lagta hai ki ye movie acha hai?
--------------------------------------------------


Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/transformers/generation/utils.py:1963: UserWarning: Using the model-agnostic default `max_length` (=21) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(


Hello, how are you doing today?
